In [2]:
import os
from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv, find_dotenv

# ----------------------------
# Config
# ----------------------------
MODEL = "gpt-5.4"   # 改成你实际用的最新模型名
REASONING_EFFORT_STEP1 = "medium"
REASONING_EFFORT_STEP2 = "medium"

BASE_DIR = Path("../")
PROMPTS_DIR = BASE_DIR / "prompts"
INPUTS_DIR = BASE_DIR / "inputs"
OUTPUTS_DIR = BASE_DIR / "outputs"

INPUT_PY = INPUTS_DIR / "HyyAnalysis.py"
PROMPT_1 = PROMPTS_DIR / "prompt_extract_reference.txt"
PROMPT_2 = PROMPTS_DIR / "prompt_generate_l123_rubric.txt"

OUT_REF = OUTPUTS_DIR / "reference_analysis.md"
OUT_RUBRIC = OUTPUTS_DIR / "l123_rubric.md"

load_dotenv(find_dotenv())
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

def read_text(path: Path) -> str:
    return path.read_text(encoding="utf-8")

def write_text(path: Path, content: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content, encoding="utf-8")

def call_model(prompt: str, effort: str) -> str:
    response = client.responses.create(
        model=MODEL,
        input=prompt,
        reasoning={"effort": effort},
        max_output_tokens=4000,
    )
    return response.output_text.strip()

In [3]:
py_text = read_text(INPUT_PY)
prompt1_template = read_text(PROMPT_1)
prompt2_template = read_text(PROMPT_2)

prompt1 = prompt1_template.replace("{{PYTHON_SCRIPT}}", py_text)
reference_md = call_model(prompt1, REASONING_EFFORT_STEP1)
write_text(OUT_REF, reference_md + "\n")
print(f"Wrote {OUT_REF}")

Wrote ../outputs/reference_analysis.md


In [4]:
import re

def split_generated_files(text: str, output_dir: Path):
    pattern = r"=== FILE: (.+?) ===\n(.*?)(?=\n=== FILE: |\Z)"
    matches = re.findall(pattern, text, flags=re.S)

    output_dir.mkdir(parents=True, exist_ok=True)

    for filename, content in matches:
        path = output_dir / filename.strip()
        path.write_text(content.strip() + "\n", encoding="utf-8")
        print(f"Wrote {path}")

In [ ]:
prompt2 = prompt2_template.replace("{{REFERENCE_ANALYSIS_MD}}", reference_md)
rubric_md = call_model(prompt2, REASONING_EFFORT_STEP2)
write_text(OUT_RUBRIC, rubric_md + "\n")
print(f"Wrote {OUT_RUBRIC}")
